## 1. Configuration et Imports

Import des bibliotheques necessaires pour la manipulation des donnees NetCDF satellites.

In [1]:
import os
import glob
import numpy as np
import pandas as pd
import xarray as xr
from datetime import datetime
import warnings
warnings.filterwarnings('ignore')

# Configuration des chemins
BASE_PATH = '/home/henintsoa/CFIM'
WAVES_PATH = os.path.join(BASE_PATH, 'data_britanique', 'waves')
OUTPUT_PATH = os.path.join(BASE_PATH, 'final/data')

print("Configuration chargee")
print(f"Chemin des donnees satellites: {WAVES_PATH}")
print(f"Chemin de sortie: {OUTPUT_PATH}")

Configuration chargee
Chemin des donnees satellites: /home/henintsoa/CFIM/data_britanique/waves
Chemin de sortie: /home/henintsoa/CFIM/final/data


## 2. Definition des Zones Cotieres de Madagascar

Definition des **bounding boxes** (boites englobantes) pour chaque zone cotiere.
Ces coordonnees correspondent aux zones utilisees dans les bulletins meteo.

Format des coordonnees: (lat_min, lat_max, lon_min, lon_max)
Note: Les latitudes sont negatives pour l'hemisphere sud

In [2]:
# Definition des zones cotieres avec leurs limites geographiques
ZONES_COTIERES = {
    # Cote EST
    "CAP D'AMBRE A TOAMASINA": {
        'lat_min': -18.5, 'lat_max': -11.5,
        'lon_min': 49.0, 'lon_max': 51.0,
        'cote': 'EST',
        'description': 'Cote nord-est, de la pointe nord jusqu\'a Toamasina'
    },
    "CAP D'AMBRE A MAHANORO": {
        'lat_min': -20.0, 'lat_max': -11.5,
        'lon_min': 48.5, 'lon_max': 51.0,
        'cote': 'EST',
        'description': 'Cote est etendue jusqu\'a Mahanoro'
    },
    "MAHANORO AU CAP SAINTE MARIE": {
        'lat_min': -25.6, 'lat_max': -19.8,
        'lon_min': 45.0, 'lon_max': 49.5,
        'cote': 'SUD-EST',
        'description': 'Cote sud-est, de Mahanoro au Cap Sainte Marie'
    },
    "TOAMASINA AU CAP SAINTE MARIE": {
        'lat_min': -25.6, 'lat_max': -18.0,
        'lon_min': 45.0, 'lon_max': 50.0,
        'cote': 'EST-SUD',
        'description': 'Toute la cote est-sud'
    },
    
    # Cote OUEST
    "CAP D'AMBRE A BESALAMPY": {
        'lat_min': -17.0, 'lat_max': -11.5,
        'lon_min': 45.0, 'lon_max': 49.5,
        'cote': 'NORD-OUEST',
        'description': 'Cote nord-ouest, de Cap d\'Ambre a Besalampy'
    },
    "BESALAMPY A MOROMBE": {
        'lat_min': -21.8, 'lat_max': -16.5,
        'lon_min': 43.0, 'lon_max': 46.0,
        'cote': 'OUEST',
        'description': 'Cote ouest centrale'
    },
    "MOROMBE AU CAP SAINTE MARIE": {
        'lat_min': -25.6, 'lat_max': -21.5,
        'lon_min': 43.0, 'lon_max': 46.0,
        'cote': 'SUD-OUEST',
        'description': 'Cote sud-ouest'
    },
    
    # Zones complementaires
    "CAP D'AMBRE A ANTALAHA": {
        'lat_min': -15.0, 'lat_max': -11.5,
        'lon_min': 48.5, 'lon_max': 50.5,
        'cote': 'NORD-EST',
        'description': 'Extreme nord-est'
    },
    "ANTALAHA A TOAMASINA": {
        'lat_min': -18.5, 'lat_max': -14.8,
        'lon_min': 49.0, 'lon_max': 50.5,
        'cote': 'EST',
        'description': 'Cote est entre Antalaha et Toamasina'
    },
    "TOAMASINA A MANANJARY": {
        'lat_min': -21.5, 'lat_max': -18.0,
        'lon_min': 47.5, 'lon_max': 50.0,
        'cote': 'EST',
        'description': 'Cote est entre Toamasina et Mananjary'
    },
    "MANANJARY A TAOLAGNARO": {
        'lat_min': -25.1, 'lat_max': -21.0,
        'lon_min': 46.5, 'lon_max': 48.5,
        'cote': 'SUD-EST',
        'description': 'Cote sud-est de Mananjary a Fort-Dauphin'
    },
    "NOSY BE ET ENVIRONS": {
        'lat_min': -14.0, 'lat_max': -12.5,
        'lon_min': 47.5, 'lon_max': 49.0,
        'cote': 'NORD-OUEST',
        'description': 'Zone de Nosy Be'
    }
}

print(f"{len(ZONES_COTIERES)} zones cotieres definies")
print("\nApercu des zones:")
for zone, info in list(ZONES_COTIERES.items())[:5]:
    print(f"  {zone} ({info['cote']})")

12 zones cotieres definies

Apercu des zones:
  CAP D'AMBRE A TOAMASINA (EST)
  CAP D'AMBRE A MAHANORO (EST)
  MAHANORO AU CAP SAINTE MARIE (SUD-EST)
  TOAMASINA AU CAP SAINTE MARIE (EST-SUD)
  CAP D'AMBRE A BESALAMPY (NORD-OUEST)


**Resultat attendu:** Liste de 12 zones cotieres avec leurs coordonnees geographiques

## 3. Inventaire des Fichiers NetCDF Disponibles

Identification de tous les fichiers de donnees satellites disponibles.

In [3]:
# Lister tous les fichiers NetCDF disponibles
nc_files = []

for year_dir in sorted(os.listdir(WAVES_PATH)):
    year_path = os.path.join(WAVES_PATH, year_dir)
    if os.path.isdir(year_path):
        for nc_file in sorted(glob.glob(os.path.join(year_path, '*.nc'))):
            filename = os.path.basename(nc_file)
            # Extraire l'annee et le mois du nom de fichier
            # Format: ESACCI-SEASTATE-L4-SWH-MULTI_1M-YYYYMM-fv01.nc
            parts = filename.split('-')
            if len(parts) >= 6:
                year_month = parts[5]  # YYYYMM
                year = int(year_month[:4])
                month = int(year_month[4:6])
                nc_files.append({
                    'chemin': nc_file,
                    'fichier': filename,
                    'annee': year,
                    'mois': month
                })

df_files = pd.DataFrame(nc_files)

print("FICHIERS NETCDF DISPONIBLES")
print("=" * 60)
print(f"\nNombre total de fichiers: {len(df_files)}")
print(f"\nRepartition par annee:")
print(df_files.groupby('annee').size().to_string())
print(f"\nPeriode couverte: {df_files['annee'].min()}-{df_files['mois'].min():02d} a {df_files['annee'].max()}-{df_files['mois'].max():02d}")

FICHIERS NETCDF DISPONIBLES

Nombre total de fichiers: 24

Repartition par annee:
annee
2017    12
2018    12

Periode couverte: 2017-01 a 2018-12


**Resultat attendu:** 24 fichiers NetCDF couvrant 2017-2018

## 4. Exploration de la Structure des Donnees NetCDF

Analyse d'un fichier exemple pour comprendre les variables disponibles.

In [4]:
# Ouvrir un fichier exemple pour explorer sa structure
sample_file = df_files.iloc[0]['chemin']

with xr.open_dataset(sample_file) as ds:
    print("STRUCTURE DU FICHIER NETCDF")
    print("=" * 60)
    
    print("\nDimensions:")
    for dim, size in ds.sizes.items():
        print(f"  {dim}: {size}")
    
    print("\nCoordonnees:")
    print(f"  Latitude: {float(ds.lat.min()):.1f} a {float(ds.lat.max()):.1f}")
    print(f"  Longitude: {float(ds.lon.min()):.1f} a {float(ds.lon.max()):.1f}")
    print(f"  Resolution: 1 x 1")
    
    print("\nVariables principales (SWH):")
    swh_vars = [v for v in ds.data_vars if 'swh' in v.lower()]
    for var in swh_vars[:5]:
        attrs = ds[var].attrs
        print(f"  {var}:")
        print(f"    - Description: {attrs.get('long_name', 'N/A')}")
        print(f"    - Unite: {attrs.get('units', 'N/A')}")

STRUCTURE DU FICHIER NETCDF

Dimensions:
  lat: 160
  nv: 2
  lon: 360
  time: 1

Coordonnees:
  Latitude: -79.5 a 79.5
  Longitude: -179.5 a 179.5
  Resolution: 1 x 1

Variables principales (SWH):
  swh_mean:
    - Description: mean of median significant wave height values
    - Unite: m
  swh_rms:
    - Description: rms of median significant wave height values
    - Unite: m
  swh_count:
    - Description: number of median significant wave height values
    - Unite: 1
  swh_sum:
    - Description: total of median significant wave height values
    - Unite: m
  swh_squared_sum:
    - Description: total of median significant wave height squared values
    - Unite: m2


In [5]:
# Visualiser les donnees pour Madagascar
# Limites approximatives de Madagascar: lat -26 a -11, lon 42 a 51

MDG_BOUNDS = {
    'lat_min': -26.0,
    'lat_max': -11.0,
    'lon_min': 42.0,
    'lon_max': 52.0
}

with xr.open_dataset(sample_file) as ds:
    # Extraire les donnees pour Madagascar
    # Verifier l'ordre des latitudes (croissant ou decroissant)
    if ds.lat[0] < ds.lat[-1]:  # Latitudes croissantes
        ds_mdg = ds.sel(
            lat=slice(MDG_BOUNDS['lat_min'], MDG_BOUNDS['lat_max']),
            lon=slice(MDG_BOUNDS['lon_min'], MDG_BOUNDS['lon_max'])
        )
    else:  # Latitudes decroissantes
        ds_mdg = ds.sel(
            lat=slice(MDG_BOUNDS['lat_max'], MDG_BOUNDS['lat_min']),
            lon=slice(MDG_BOUNDS['lon_min'], MDG_BOUNDS['lon_max'])
        )
    
    print("DONNEES EXTRAITES POUR MADAGASCAR")
    print("=" * 60)
    print(f"\nDimensions apres extraction:")
    print(f"  Latitude: {len(ds_mdg.lat)} points")
    print(f"  Longitude: {len(ds_mdg.lon)} points")
    
    swh_data = ds_mdg['swh_mean'].values[0]  # Premier (et seul) pas de temps
    valid_data = swh_data[~np.isnan(swh_data)]
    
    print(f"\nStatistiques SWH (hauteur de vague):")
    print(f"  Minimum: {np.nanmin(swh_data):.2f} m")
    print(f"  Maximum: {np.nanmax(swh_data):.2f} m")
    print(f"  Moyenne: {np.nanmean(swh_data):.2f} m")
    print(f"  Points de donnees valides: {len(valid_data)} / {swh_data.size}")

DONNEES EXTRAITES POUR MADAGASCAR

Dimensions apres extraction:
  Latitude: 15 points
  Longitude: 10 points

Statistiques SWH (hauteur de vague):
  Minimum: 0.63 m
  Maximum: 9969209968386869046778552952102584320.00 m
  Moyenne: 2525533191991340040458071342791524352.00 m
  Points de donnees valides: 150 / 150


**Resultat attendu:** Statistiques globales de hauteur de vague pour Madagascar

## 5. Fonction d'Extraction SWH par Zone Cotiere

Creation des fonctions pour extraire et calculer les statistiques SWH pour chaque zone cotiere.

In [6]:
def extraire_swh_zone(ds, zone_name, zone_bounds):
    """
    Extrait les statistiques SWH pour une zone cotiere donnee.
    
    Parameters:
    -----------
    ds : xarray.Dataset
        Dataset NetCDF ouvert contenant les donnees SWH
    zone_name : str
        Nom de la zone cotiere
    zone_bounds : dict
        Dictionnaire avec lat_min, lat_max, lon_min, lon_max
    
    Returns:
    --------
    dict : Statistiques SWH pour la zone (moyenne, min, max, ecart-type)
    """
    try:
        # Extraire les limites de la zone
        lat_min, lat_max = zone_bounds['lat_min'], zone_bounds['lat_max']
        lon_min, lon_max = zone_bounds['lon_min'], zone_bounds['lon_max']
        
        # Selectionner la zone geographique
        # Attention: verifier l'ordre des latitudes dans le fichier
        if ds.lat[0] > ds.lat[-1]:  # Latitudes decroissantes
            ds_zone = ds.sel(
                lat=slice(lat_max, lat_min),
                lon=slice(lon_min, lon_max)
            )
        else:  # Latitudes croissantes
            ds_zone = ds.sel(
                lat=slice(lat_min, lat_max),
                lon=slice(lon_min, lon_max)
            )
        
        # Extraire les donnees SWH
        swh_mean = ds_zone['swh_mean'].values
        swh_max = ds_zone['swh_max'].values if 'swh_max' in ds_zone else None
        
        # Calculer les statistiques (en ignorant les valeurs NaN)
        valid_data = swh_mean[~np.isnan(swh_mean)]
        
        # Verifier qu'il y a des donnees valides
        if len(valid_data) == 0:
            return None
        
        # Construire le dictionnaire de resultats
        result = {
            'zone': zone_name,
            'swh_mean': float(np.nanmean(swh_mean)),
            'swh_min': float(np.nanmin(swh_mean)),
            'swh_max_value': float(np.nanmax(swh_mean)),
            'swh_std': float(np.nanstd(swh_mean)),
            'n_points': len(valid_data),
            'cote': zone_bounds['cote']
        }
        
        # Ajouter le maximum absolu si disponible
        if swh_max is not None:
            result['swh_max_absolu'] = float(np.nanmax(swh_max))
        
        return result
        
    except Exception as e:
        print(f"  Erreur pour {zone_name}: {str(e)}")
        return None


def traiter_fichier_nc(nc_path, zones):
    """
    Traite un fichier NetCDF et extrait les SWH pour toutes les zones.
    
    Parameters:
    -----------
    nc_path : str
        Chemin vers le fichier NetCDF
    zones : dict
        Dictionnaire des zones cotieres
    
    Returns:
    --------
    list : Liste des resultats pour chaque zone
    """
    results = []
    
    try:
        # Ouvrir le fichier NetCDF
        with xr.open_dataset(nc_path) as ds:
            # Extraire la date du fichier
            time_val = pd.Timestamp(ds.time.values[0])
            
            # Traiter chaque zone cotiere
            for zone_name, zone_bounds in zones.items():
                zone_result = extraire_swh_zone(ds, zone_name, zone_bounds)
                
                if zone_result:
                    # Ajouter les informations temporelles
                    zone_result['date'] = time_val
                    zone_result['annee'] = time_val.year
                    zone_result['mois'] = time_val.month
                    results.append(zone_result)
                    
    except Exception as e:
        print(f"Erreur fichier {nc_path}: {str(e)}")
    
    return results

print("Fonctions d'extraction definies")

Fonctions d'extraction definies


## 6. Extraction des Donnees SWH pour Toutes les Zones

Traitement de tous les fichiers NetCDF pour extraire les donnees de hauteur de vague.

In [7]:
# Traiter tous les fichiers NetCDF
print("EXTRACTION DES DONNEES SWH")
print("=" * 60)

all_results = []

# Boucle sur tous les fichiers
for idx, row in df_files.iterrows():
    print(f"\rTraitement: {row['annee']}-{row['mois']:02d}...", end="")
    
    # Extraire les donnees pour toutes les zones
    results = traiter_fichier_nc(row['chemin'], ZONES_COTIERES)
    all_results.extend(results)

print(f"\n\nExtraction terminee!")
print(f"{len(all_results)} enregistrements extraits")

# Creer le DataFrame
df_swh = pd.DataFrame(all_results)
print(f"\nDimensions du dataset: {df_swh.shape}")
print(f"\nColonnes: {list(df_swh.columns)}")

EXTRACTION DES DONNEES SWH
Traitement: 2018-12...

Extraction terminee!
288 enregistrements extraits

Dimensions du dataset: (288, 11)

Colonnes: ['zone', 'swh_mean', 'swh_min', 'swh_max_value', 'swh_std', 'n_points', 'cote', 'swh_max_absolu', 'date', 'annee', 'mois']


Extraction terminee!
288 enregistrements extraits

Dimensions du dataset: (288, 11)

Colonnes: ['zone', 'swh_mean', 'swh_min', 'swh_max_value', 'swh_std', 'n_points', 'cote', 'swh_max_absolu', 'date', 'annee', 'mois']


In [8]:
# Afficher un apercu des donnees extraites
print("APERCU DES DONNEES EXTRAITES")
print("=" * 60)

# Statistiques par zone
print("\nStatistiques SWH par zone cotiere:")
stats_zone = df_swh.groupby('zone').agg({
    'swh_mean': ['mean', 'min', 'max', 'std'],
    'n_points': 'mean'
}).round(2)
stats_zone.columns = ['SWH_moy (m)', 'SWH_min (m)', 'SWH_max (m)', 'Ecart-type', 'Points_moy']
print(stats_zone.to_string())

# Statistiques par mois (saisonnalite)
print("\nSaisonnalite des vagues (SWH moyen par mois):")
stats_mois = df_swh.groupby('mois')['swh_mean'].mean().round(2)
print(stats_mois.to_string())

APERCU DES DONNEES EXTRAITES

Statistiques SWH par zone cotiere:
                                SWH_moy (m)   SWH_min (m)   SWH_max (m)    Ecart-type  Points_moy
zone                                                                                             
ANTALAHA A TOAMASINA           1.090382e+36  1.270000e+00  2.492302e+36  9.936988e+35         8.0
BESALAMPY A MOROMBE            4.453837e+36  3.876915e+36  5.538450e+36  3.046330e+35        18.0
CAP D'AMBRE A ANTALAHA         2.423072e+36  1.661535e+36  2.492302e+36  2.345505e+35        12.0
CAP D'AMBRE A BESALAMPY        3.212301e+36  2.658456e+36  3.655377e+36  2.713275e+35        30.0
CAP D'AMBRE A MAHANORO         3.046147e+36  2.584610e+36  3.692300e+36  3.127340e+35        27.0
CAP D'AMBRE A TOAMASINA        1.739419e+36  1.246151e+36  2.492302e+36  5.190039e+35        16.0
MAHANORO AU CAP SAINTE MARIE   4.029222e+36  3.987684e+36  4.319991e+36  1.122639e+35        30.0
MANANJARY A TAOLAGNARO         4.223068e+36  4.153837

**Resultat attendu:** Statistiques par zone montrant la variabilite spatiale et temporelle des vagues

## 7. Analyse de la Variabilite Spatiale et Temporelle

Identification des zones et periodes les plus dangereuses.

In [9]:
print("ANALYSE DE LA VARIABILITE")
print("=" * 60)

# Zones avec les vagues les plus hautes
print("\nTop 5 zones avec les vagues les plus hautes (en moyenne):")
top_zones = df_swh.groupby('zone')['swh_mean'].mean().sort_values(ascending=False).head(5)
for i, (zone, swh) in enumerate(top_zones.items(), 1):
    print(f"  {i}. {zone}: {swh:.2f} m")

# Mois les plus dangereux
print("\nMois avec les vagues les plus hautes:")
mois_names = ['Jan', 'Fev', 'Mar', 'Avr', 'Mai', 'Jun', 'Jul', 'Aou', 'Sep', 'Oct', 'Nov', 'Dec']
top_mois = df_swh.groupby('mois')['swh_mean'].mean().sort_values(ascending=False)
for mois, swh in top_mois.head(5).items():
    print(f"  {mois_names[mois-1]}: {swh:.2f} m")

# Analyse par cote
print("\nSWH moyen par type de cote:")
stats_cote = df_swh.groupby('cote')['swh_mean'].agg(['mean', 'max']).round(2)
stats_cote.columns = ['Moyenne (m)', 'Maximum (m)']
print(stats_cote.sort_values('Moyenne (m)', ascending=False).to_string())

ANALYSE DE LA VARIABILITE

Top 5 zones avec les vagues les plus hautes (en moyenne):
  1. MOROMBE AU CAP SAINTE MARIE: 5316911983139663491615228241121378304.00 m
  2. TOAMASINA AU CAP SAINTE MARIE: 4880759047022737970818666549466890240.00 m
  3. TOAMASINA A MANANJARY: 4569221235510648313106836769713684480.00 m
  4. BESALAMPY A MOROMBE: 4453836860876541294826519232933527552.00 m
  5. MANANJARY A TAOLAGNARO: 4223068111608326667970073800667561984.00 m

Mois avec les vagues les plus hautes:
  Oct: 3449800494268750758037975267224322048.00 m
  Mar: 3389416004876901630977767485443407872.00 m
  Jan: 3329031515485051913621749344956841984.00 m
  Nov: 3326916135283426700928023640334663680.00 m
  Fev: 3311916166580992623268755460115660800.00 m

SWH moyen par type de cote:
             Moyenne (m)   Maximum (m)
cote                                  
SUD-OUEST   5.316912e+36  5.316912e+36
EST-SUD     4.880759e+36  4.984605e+36
OUEST       4.453837e+36  5.538450e+36
SUD-EST     4.126145e+36  4.984605

**Resultat attendu:** Identification des zones et periodes a risque eleve

## 8. Creation du Dataset Final

Preparation du dataset avec classification de l'etat de mer et score de risque.

In [10]:
# Preparer le dataset final
print("PREPARATION DU DATASET FINAL")
print("=" * 60)

# Copie du DataFrame
df_swh_final = df_swh.copy()

# Formater la date
df_swh_final['date_str'] = df_swh_final['date'].dt.strftime('%Y-%m')

# Fonctions de classification
def classifier_swh(swh):
    """
    Classifie le niveau de risque selon la hauteur de vague.
    Basee sur l'echelle de Douglas.
    
    Echelle:
    - 0-0.5m: Calme
    - 0.5-1.25m: Peu agitee
    - 1.25-2.5m: Agitee
    - 2.5-4.0m: Forte
    - >4.0m: Tres forte
    """
    if pd.isna(swh):
        return 'Non disponible'
    elif swh < 0.5:
        return 'Calme'
    elif swh < 1.25:
        return 'Peu agitee'
    elif swh < 2.5:
        return 'Agitee'
    elif swh < 4.0:
        return 'Forte'
    else:
        return 'Tres forte'

def score_risque_swh(swh):
    """
    Calcule un score de risque (0-10) base sur SWH.
    Score lineaire: 0m = 0, 5m = 10
    """
    if pd.isna(swh):
        return np.nan
    return min(10, swh * 2)

# Appliquer les classifications
df_swh_final['etat_mer_satellite'] = df_swh_final['swh_mean'].apply(classifier_swh)
df_swh_final['score_risque_vague'] = df_swh_final['swh_mean'].apply(score_risque_swh).round(1)

# Afficher le resume
print("\nColonnes du dataset final:")
for col in df_swh_final.columns:
    print(f"  {col}")

print("\nDistribution des etats de mer:")
print(df_swh_final['etat_mer_satellite'].value_counts().to_string())

PREPARATION DU DATASET FINAL

Colonnes du dataset final:
  zone
  swh_mean
  swh_min
  swh_max_value
  swh_std
  n_points
  cote
  swh_max_absolu
  date
  annee
  mois
  date_str
  etat_mer_satellite
  score_risque_vague

Distribution des etats de mer:
etat_mer_satellite
Tres forte    255
Peu agitee     23
Agitee          9
Calme           1


In [11]:
# Afficher un apercu du dataset final
print("APERCU DU DATASET FINAL")
print("=" * 60)

colonnes_affichage = ['date_str', 'zone', 'cote', 'swh_mean', 'swh_min', 'swh_max_value', 
                      'etat_mer_satellite', 'score_risque_vague']
print(df_swh_final[colonnes_affichage].head(15).to_string())

APERCU DU DATASET FINAL
   date_str                           zone        cote      swh_mean   swh_min  swh_max_value etat_mer_satellite  score_risque_vague
0   2017-01        CAP D'AMBRE A TOAMASINA         EST  1.869227e+36  1.150993   9.969210e+36         Tres forte                10.0
1   2017-01         CAP D'AMBRE A MAHANORO         EST  2.953840e+36  0.692264   9.969210e+36         Tres forte                10.0
2   2017-01   MAHANORO AU CAP SAINTE MARIE     SUD-EST  3.987684e+36  1.406636   9.969210e+36         Tres forte                10.0
3   2017-01  TOAMASINA AU CAP SAINTE MARIE     EST-SUD  4.735375e+36  1.323358   9.969210e+36         Tres forte                10.0
4   2017-01        CAP D'AMBRE A BESALAMPY  NORD-OUEST  3.323070e+36  0.631359   9.969210e+36         Tres forte                10.0
5   2017-01            BESALAMPY A MOROMBE       OUEST  4.430760e+36  0.688779   9.969210e+36         Tres forte                10.0
6   2017-01    MOROMBE AU CAP SAINTE MARIE   

**Resultat attendu:** Dataset avec classification de l'etat de mer et score de risque

## 9. Sauvegarde des Donnees

Export du fichier final `donnees_satellites_swh.csv`

In [12]:
# Sauvegarder le dataset
output_file = os.path.join(OUTPUT_PATH, 'donnees_satellites_swh.csv')

# Selectionner les colonnes a exporter
colonnes_export = [
    'date', 'date_str', 'annee', 'mois', 'zone', 'cote',
    'swh_mean', 'swh_min', 'swh_max_value', 'swh_std',
    'n_points', 'etat_mer_satellite', 'score_risque_vague'
]

# Ajouter la colonne max absolu si elle existe
if 'swh_max_absolu' in df_swh_final.columns:
    colonnes_export.insert(10, 'swh_max_absolu')

df_export = df_swh_final[colonnes_export].copy()

# Convertir la date en string pour l'export CSV
df_export['date'] = df_export['date'].dt.strftime('%Y-%m-%d')

# Sauvegarder en CSV
df_export.to_csv(output_file, index=False, encoding='utf-8')

print("SAUVEGARDE EFFECTUEE")
print("=" * 60)
print(f"\nFichier sauvegarde: {output_file}")
print(f"Nombre d'enregistrements: {len(df_export)}")
print(f"Periode: {df_export['annee'].min()}-{df_export['mois'].min():02d} a {df_export['annee'].max()}-{df_export['mois'].max():02d}")
print(f"Zones couvertes: {df_export['zone'].nunique()}")

SAUVEGARDE EFFECTUEE

Fichier sauvegarde: /home/henintsoa/CFIM/final/data/donnees_satellites_swh.csv
Nombre d'enregistrements: 288
Periode: 2017-01 a 2018-12
Zones couvertes: 12


**Fichier produit:** `/home/henintsoa/CFIM/final/data/donnees_satellites_swh.csv`

**Structure:**
- `date`: Date de l'observation
- `zone`: Zone cotiere
- `swh_mean`: Hauteur de vague moyenne (m)
- `swh_min`: Hauteur de vague minimale (m)
- `swh_max_value`: Hauteur de vague maximale (m)
- `swh_std`: Ecart-type
- `etat_mer_satellite`: Classification de l'etat de mer
- `score_risque_vague`: Score de risque (0-10)